# 实践项目 03：脑膜瘤 H&E 形态代理分类

本 Notebook 使用课程准备的真实脑膜瘤 H&E 图块数据，完成数据核对、形态特征分类、原图级测试和染色变化比较。标签按核密度代理分数分为三个类别，只用于教学分类，不对应临床病理分级。测试集的 96 个图块仅来自 2 张原图，且代理标签与输入形态特征高度相关；图块上没有错分也不能作为独立病理分类性能。

Kaggle 是本项目的首选实践入口。打开公开 Notebook 后，点击“复制并编辑”保存到自己的账户，再按单元格顺序运行。下载 Notebook 到电脑运行是补充方式。

代码中用整行注释标出了需要填写的位置。先阅读当前单元格的输入、处理和输出，再修改标记区域。

## 任务总览

1. 找到固定的课程 NPZ，并核对图块 shape、标签、原图编号、坐标和数据划分。
2. 查看 RGB 统计与 H&E 形态特征，理解标签来源并建立简单比较。
3. 补全随机森林分类器；输入只使用图像测得的形态特征，不使用 `proxy_score`。
4. 用验证集选择树数量，在独立测试集输出混淆矩阵、宏平均 F1 和错误图块。
5. 观察颜色变化后模型的表现，并说明代理标签的结果边界。

## 需要保存的结果

`task3_data_visualization.png`、`task3_training_curve.png`、`task3_prediction_visualization.png`、`task3_pytorch_result.json`。


In [ ]:
from pathlib import Path  # 导入当前步骤需要的工具
import json, random  # 导入当前步骤需要的工具
import numpy as np  # 导入当前步骤需要的工具
import matplotlib.pyplot as plt  # 导入当前步骤需要的工具
from sklearn.ensemble import RandomForestClassifier  # 导入随机森林分类器
from sklearn.linear_model import LogisticRegression  # 导入颜色统计基线
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix  # 导入评价指标

SEED = 42  # 固定随机状态以便复现实验
random.seed(SEED); np.random.seed(SEED)  # 固定随机状态以便复现实验
INPUT = Path('/kaggle/input')  # 保存当前步骤使用的中间结果
OUT = Path('/kaggle/working'); OUT.mkdir(exist_ok=True)  # 保存输出文件目录
DATA_PATH = None  # 可在本地填写课程 NPZ 路径；Kaggle 自动查找固定文件名
candidates = sorted(INPUT.rglob('meningioma_public_morphology_tiles.npz'))  # 读取本任务需要的数据
if DATA_PATH is None and candidates:  # 根据当前条件选择处理分支
    DATA_PATH = candidates[0]  # 保存当前步骤使用的中间结果
assert DATA_PATH is not None, '请挂载包含 meningioma_public_morphology_tiles.npz 的课程数据集。'  # 执行当前步骤并保留结果
data = np.load(DATA_PATH, allow_pickle=True)  # 读取本任务需要的数据
required = {'images','labels','proxy_scores','morphology_features','feature_names','source_image_ids','coordinates_yx','split'}  # 核对输入字段
assert required.issubset(data.files), required - set(data.files)  # 执行当前步骤并保留结果
images = data['images']; labels = data['labels'].astype(np.int64)  # 读取图像与标签
proxy_scores = data['proxy_scores'].astype(np.float32)  # 读取连续代理分数，仅用于说明标签来源
morphology_features = data['morphology_features'].astype(np.float32)  # 读取图像测得的形态统计
feature_names = data['feature_names'].astype(str)  # 读取形态特征名称
source_image_ids = data['source_image_ids'].astype(str)  # 读取图块对应的原图编号
coordinates_yx = data['coordinates_yx'].astype(np.int32)  # 读取图块在原图中的左上角坐标
split = data['split'].astype(str)  # 读取预先划分的数据集合
print('data:', images.shape, 'splits:', {k: int((split == k).sum()) for k in np.unique(split)})  # 显示核对结果


## 任务 1：完成数据核对

输入包含图像、标签、连续代理分数、形态统计、原图编号、坐标和 `split`。请输出每个 split 的图块数、原图编号、坐标范围和三类标签数量。


In [ ]:
# ===== 项目03·任务1·学生填写区（开始） =====
# TODO：根据 images、labels、source_image_ids、coordinates_yx 和 split 生成 summary。
summary = None  # 保存当前步骤的核对结果
# ===== 项目03·任务1·学生填写区（结束） =====
print(summary)  # 显示便于检查的关键信息


## 任务 2：查看图像统计与形态特征

`morphology_features` 的第一列 `proxy_score` 只用于说明标签生成规则，不能作为模型输入；其余五列由 H&E 图像测得，可以用于观察核相关信号、深色比例和染色统计。


In [ ]:
# ===== 项目03·任务2·学生填写区（开始） =====
# TODO：输出训练集的 feature_names[1:]、均值和标准差，并用 RGB 均值建立一个颜色统计比较。
usable_features = morphology_features[:, 1:]  # 排除由标签规则得到的 proxy_score
feature_summary = None  # 保存当前步骤的核对结果
# ===== 项目03·任务2·学生填写区（结束） =====
print(feature_summary)  # 显示便于检查的关键信息


## 任务 3：补全形态特征随机森林

训练数据来自不同原图。模型输入为五个图像测得的形态特征，输出三个代理标签；`proxy_score` 已明确排除，避免把标签生成规则直接喂给模型。


In [ ]:
# ===== 项目03·任务3·学生填写区（开始） =====
# TODO：补全随机森林参数，使用 morphology_features[:, 1:] 训练三分类模型。
train_idx = np.where(split == 'train')[0]  # 取出训练图块
val_idx = np.where(split == 'validation')[0]  # 取出验证图块
test_idx = np.where(split == 'test')[0]  # 取出测试图块
model = None  # 保存当前步骤的模型
# ===== 项目03·任务3·学生填写区（结束） =====
print(model)  # 显示便于检查的模型结构


## 任务 4：验证集选模与独立测试

比较不同树数量的验证集宏平均 F1，固定最佳设置后只在测试集评价一次，并显示混淆矩阵和错误图块。


In [ ]:
# ===== 项目03·任务4·学生填写区（开始） =====
# TODO：完成树数量比较、验证集选模、测试评价和混淆矩阵。
# ===== 项目03·任务4·学生填写区（结束） =====
print('请先完成验证集选模和测试评价。')  # 显示便于检查的提示


## 任务 5：错误图块与染色变化

查看测试集错误图块，固定模型后改变 RGB 通道，再比较宏平均 F1。最后写出代理标签、图像来源和模型评价各自能支持的判断。


In [ ]:
# ===== 项目03·任务5·学生填写区（开始） =====
# TODO：完成错误图块可视化、染色变化比较和 JSON 保存。
# ===== 项目03·任务5·学生填写区（结束） =====
print('输出文件应写入', OUT)  # 显示便于检查的关键信息
